# Experiment 3 (final) — Physics-informed loss for low-data mammogram DDPM: multi-seed, mammography-domain metrics, downstream utility
**Question:** in a data-scarce setting (25% of CBIS-DDSM training data), does adding a physics-informed consistency loss
to a published conditional DDPM recipe improve synthetic mammograms?

**Design (fixes every weakness of Exp 1-2):** 3 training seeds per arm · identical init/data/noise per seed · physics term is a
two-sided band calibrated on real images · Inception KID/FID **plus** mammography-space KID and precision/recall (oracle features) ·
**downstream test**: does adding synthetic images to the 25% real set improve a ResNet-18 benign/malignant classifier on the real test set?

**Pre-registered (fixed BEFORE running):**
* PRIMARY: downstream AUC with PINN synthetic > with baseline synthetic, Welch t-test p < 0.05.
* SECONDARY: Inception KID lower in all 3 seeds; physics residual at least as close to real.
* All results are reported whichever way they come out.

**How to run**
* **Session A (train):** Cell 1 `MODE="train"`, set `BUDGET_H`, GPU T4 ×2, Internet ON, inputs CBIS-DDSM + oracle-checkpoint-final → *Save & Run All (Commit)*.
* **Session B (eval):** Cell 1 `MODE="eval"`, additionally attach Session A's output (Add Input → Your Work) → *Save & Run All (Commit)*.
* Cells 2-3 run in both modes; 4-5 only in train mode; 6-8 only in eval mode.

In [ ]:
# ===================== CELL 1 — CONFIG (the ONLY cell you ever edit) =====================
MODE = "train"        # "train" = Session A: smoke test + training of all seeds
                      # "eval"  = Session B: evaluation + downstream classifier test + final statistics
BUDGET_H = 8.5        # train mode only: hours this session may train (>= 0.8 h BELOW remaining GPU quota!)
SEEDS = [42, 43, 44]  # 3 independent training seeds per arm (fixes the single-seed weakness)
EPOCHS = 45           # per run: 45 epochs x ~78 optimizer steps (25% data) = ~3,500 optimizer steps
DATA_FRAC = 0.25      # low-data regime: 25% of each class (~620 real images)
LAM = 0.01            # physics weight of the PINN arm
import os, time
if not os.path.exists("/tmp/pg_session_start"):
    open("/tmp/pg_session_start", "w").write(str(time.time()))
assert MODE in ("train", "eval"), "MODE must be 'train' or 'eval'"
print(f"MODE={MODE} | seeds={SEEDS} | epochs={EPOCHS} | data_frac={DATA_FRAC} | lambda={LAM} | budget={BUDGET_H} h")


In [ ]:
# ===================== CELL 2 — setup (every session; needs Internet ON) (run in EVERY new session; needs Internet ON) =====================
import os, re, subprocess, time
if not os.path.exists("/tmp/pg_session_start"):
    open("/tmp/pg_session_start", "w").write(str(time.time()))
os.chdir("/kaggle/working")
if not os.path.isdir("/kaggle/working/U24_synthesis"):
    subprocess.run("git clone -q https://github.com/Its-Me-Sk/U24_synthesis.git", shell=True, check=True)
if not os.path.exists("/tmp/pip_done"):
    subprocess.run("cd /kaggle/working/U24_synthesis && grep -v xformers requirements.txt > /tmp/req.txt && "
                   "pip install -q -r /tmp/req.txt && "
                   "pip install -q numpy==1.26.4 pandas==2.2.2 --force-reinstall && "
                   "pip install -q --no-deps torch-fidelity", shell=True, check=True)
    open("/tmp/pip_done", "w").write("ok")
p = "/kaggle/working/U24_synthesis/lightning_synthesis/model_architectures/monai_ddpm.py"
s = open(p).read()
s2 = re.sub(r"use_flash_attention\s*=\s*True", "use_flash_attention=False", s)   # xformers not installed
if s2 != s:
    open(p, "w").write(s2)
# models.py (oracle loader) has a hard-coded path from the original author's machine -> point it to the repo's config.yaml
m = "/kaggle/working/U24_synthesis/models.py"
s = open(m).read()
s2 = re.sub(r"open\(\s*['\"]/home/[^'\"]*config\.yaml['\"]", "open('/kaggle/working/U24_synthesis/config.yaml'", s)
if s2 != s:
    open(m, "w").write(s2)
print("models.py config path ->", "patched" if s2 != s else "already ok")
print("Cell 1 done. Kernel restart NOT needed (all GPU work runs in subprocesses).")


In [ ]:
# ===================== CELL 3 — data cache + code + GPU check (every session, no training) =====================
import os, sys, subprocess, threading, time
from pathlib import Path
WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
os.chdir(WD)

Path("/tmp/prepare.py").write_text(r'''
import os, subprocess
import numpy as np, pandas as pd
from PIL import Image
if os.path.exists("/tmp/train_cache.npz") and os.path.exists("/tmp/test_cache.npz"):
    print("caches already built"); raise SystemExit(0)
hit = subprocess.run("find /kaggle/input -maxdepth 6 -name dicom_info.csv", shell=True,
                     capture_output=True, text=True).stdout.split()
if not hit:
    raise SystemExit("CBIS-DDSM not attached: Add Input -> CBIS-DDSM Breast Cancer Image Dataset")
RAW = os.path.dirname(os.path.dirname(hit[0]))
info = pd.read_csv(os.path.join(RAW, "csv/dicom_info.csv"))
lookup = info[info["SeriesDescription"] == "full mammogram images"].set_index("PatientName")["image_path"].to_dict()
for split, csvs in [("train", ["csv/calc_case_description_train_set.csv", "csv/mass_case_description_train_set.csv"]),
                    ("test",  ["csv/calc_case_description_test_set.csv",  "csv/mass_case_description_test_set.csv"])]:
    xs, skipped = {0: [], 1: []}, 0
    for c in csvs:
        for _, row in pd.read_csv(os.path.join(RAW, c)).iterrows():
            lab = 1 if row["pathology"] == "MALIGNANT" else 0
            rel = lookup.get(row["image file path"].split("/")[0])
            if rel is None:
                skipped += 1; continue
            p = os.path.join(RAW, "jpeg", rel.replace("CBIS-DDSM/jpeg/", ""))
            if not os.path.exists(p):
                skipped += 1; continue
            xs[lab].append(np.array(Image.open(p).convert("L").resize((256, 256)), dtype=np.uint8))
    if split == "train" and len(xs[0]) > 1300:          # same balancing as before: 1300 benign, all malignant
        keep = np.sort(np.random.default_rng(42).choice(len(xs[0]), 1300, replace=False))
        xs[0] = [xs[0][i] for i in keep]
    x = np.stack(xs[0] + xs[1]); y = np.array([0] * len(xs[0]) + [1] * len(xs[1]), dtype=np.int64)
    np.savez(f"/tmp/{split}_cache.npz", x=x, y=y)
    print(f"{split}: {x.shape} benign={int((y == 0).sum())} malignant={int((y == 1).sum())} skipped={skipped}")
''')
subprocess.run([sys.executable, "/tmp/prepare.py"], check=True)

Path("pinn_physics.py").write_text(r'''# pinn_physics.py - physics-informed soft constraint (PINN-style residual) for mammogram DDPM.
# Physical model: signal-dependent noise-level function (NLF) of the X-ray film/digitiser chain
#   sigma^2(mu) = b + a*mu + c*mu^2
#   b: Gaussian read/digitiser noise, a*mu: Poisson (quantum) noise, c*mu^2: multiplicative film-grain/gain term
# The coefficients are CALIBRATED on real training images, then the residual between the NLF of a
# generated image and the physical NLF is added to the diffusion loss (lambda * residual).
import torch
import torch.nn.functional as F


class PhysicsNLF:
    def __init__(self, device, k=7, K=16, lo=0.08, hi=0.95, mask_lo=0.04):
        ax = torch.arange(7, device=device, dtype=torch.float32) - 3
        g = torch.exp(-ax ** 2 / 2.0)
        self.g = g / g.sum()
        self.k, self.K, self.lo, self.hi, self.mask_lo = k, K, lo, hi, mask_lo
        self.c = torch.linspace(lo, hi, K, device=device)
        self.bw = float(hi - lo) / (K - 1) * 0.6
        self.p = None                     # (b, a, c) of the calibrated NLF

    def blur(self, I):
        g = self.g
        I = F.conv2d(F.pad(I, (3, 3, 0, 0), mode="reflect"), g.view(1, 1, 1, 7))
        return F.conv2d(F.pad(I, (0, 0, 3, 3), mode="reflect"), g.view(1, 1, 7, 1))

    def stats(self, I):
        # I: (B,1,H,W) intensities in [0,1]. Returns per-image binned high-frequency variance vs local mean.
        h = I - self.blur(I)
        mu = F.avg_pool2d(I, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        v = F.avg_pool2d(h * h, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        W = torch.exp(-(mu.unsqueeze(-1) - self.c) ** 2 / (2 * self.bw ** 2))
        W = W * (mu > self.mask_lo).unsqueeze(-1).float()        # ignore black background (air)
        mass = W.sum(1)                                           # (B,K)
        vbin = torch.einsum("bpk,bp->bk", W, v) / (mass + 1e-6)   # (B,K)
        return vbin, mass

    def model_var(self):
        b, a, c = self.p
        return (b + a * self.c + c * self.c ** 2).clamp_min(1e-7)

    def residual(self, I):
        # per-image physics residual: mass-weighted squared log-ratio of measured vs physical NLF
        vbin, mass = self.stats(I)
        r = (torch.log(vbin + 1e-5) - torch.log(self.model_var() + 1e-5)) ** 2
        w = mass / (mass.sum(1, keepdim=True) + 1e-6)
        return (w * r).sum(1)

    @torch.no_grad()
    def fit(self, X_u8, bs=32):
        dev = self.c.device
        tv = torch.zeros(self.K, device=dev, dtype=torch.float64)
        tm = torch.zeros(self.K, device=dev, dtype=torch.float64)
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            vb, m = self.stats(I)
            tv += (vb * m).sum(0).double()
            tm += m.sum(0).double()
        V = tv / tm.clamp_min(1e-6)
        wt = tm / tm.sum()
        c = self.c.double()

        def wls(cols):
            A = torch.stack(cols, 1)
            th = torch.linalg.solve(A.T @ (wt[:, None] * A), A.T @ (wt * V))
            pred = (A @ th).clamp_min(1e-9)
            ss_res = (wt * (V - pred) ** 2).sum()
            ss_tot = (wt * (V - (wt * V).sum()) ** 2).sum()
            lr = torch.log(V) - torch.log(pred)
            lres = (wt * lr ** 2).sum()
            ltot = (wt * (torch.log(V) - (wt * torch.log(V)).sum()) ** 2).sum()
            return th, float(1 - ss_res / ss_tot), float(1 - lres / ltot)

        th_lin, r2_lin, lr2_lin = wls([torch.ones_like(c), c])
        th_q, r2_q, lr2_q = wls([torch.ones_like(c), c, c ** 2])
        self.p = (float(th_q[0]), float(th_q[1]), float(th_q[2]))
        return {"model": "sigma^2 = b + a*mu + c*mu^2 (Poisson-Gaussian + multiplicative)",
                "b": self.p[0], "a": self.p[1], "c": self.p[2],
                "R2_var": r2_q, "R2_logvar": lr2_q,
                "linear_poisson_gaussian_only": {"b": float(th_lin[0]), "a": float(th_lin[1]),
                                                  "R2_var": r2_lin, "R2_logvar": lr2_lin},
                "bin_centers": [float(x) for x in c], "measured_var": [float(x) for x in V],
                "bin_weight": [float(x) for x in wt]}

    @torch.no_grad()
    def mean_residual(self, X_u8, bs=32):
        dev = self.c.device
        tot, n = 0.0, 0
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            tot += float(self.residual(I).sum())
            n += I.shape[0]
        return tot / max(n, 1)

    def state(self):
        return {"p": list(self.p), "k": self.k, "K": self.K, "lo": self.lo, "hi": self.hi, "mask_lo": self.mask_lo}

    def load(self, s):
        self.p = tuple(s["p"])
''')
Path("train_arm3.py").write_text(r'''# train_arm3.py - Experiment 3: LOW-DATA regime + two-sided physics BAND constraint, on the ORIGINAL recipe.
# Both arms train on the same stratified subset (data_frac) of CBIS-DDSM train. The physics term penalizes the
# per-image residual only when it leaves the [q_lo, q_hi] band of REAL images (too unphysical OR too regular).
# Same as the original: 0.7*MSE + 0.2*(1-MS-SSIM) + 0.05*HF loss on x0_hat, per-image min-max to [-1,1],
# shear/scale/flip augmentation, 15% label dropout (-1), batch 2 x grad-accum 4, Adam, clip 1.0, fp16.
# The ONLY difference between arms: PINN adds lam * physics-consistency hinge term.
import os, sys, glob, time, json, argparse
import numpy as np
import torch
import torch.nn.functional as F
from torchmetrics.functional.image import multiscale_structural_similarity_index_measure as msssim

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
sys.path.insert(0, WD)
from pinn_physics import PhysicsNLF

ap = argparse.ArgumentParser()
ap.add_argument("--arm", required=True)                  # base3 | pinn3
ap.add_argument("--data_frac", type=float, default=0.25) # fraction of each class used for training
ap.add_argument("--band_lo", type=float, default=0.1)    # lower band edge = this quantile of REAL residuals
ap.add_argument("--band_hi", type=float, default=0.9)    # upper band edge = this quantile of REAL residuals
ap.add_argument("--lam", type=float, default=0.0)        # physics weight, 0 = baseline
ap.add_argument("--epochs", type=int, default=400)
ap.add_argument("--bs", type=int, default=2)
ap.add_argument("--accum", type=int, default=4)
ap.add_argument("--lr", type=float, default=1e-5)
ap.add_argument("--p_uncond", type=float, default=0.15)
ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--budget_h", type=float, default=10.9)
ap.add_argument("--snap_every", type=int, default=20)
ap.add_argument("--smoke_batches", type=int, default=0)
ap.add_argument("--log_every", type=int, default=200)
ap.add_argument("--ckpt_every", type=int, default=1)     # full (resumable) checkpoint every N epochs
ap.add_argument("--final", action="store_true")          # on completion: write <arm>_done.json, delete big resume files
ap.add_argument("--ckpt_dir", default="/kaggle/working/ckpt")
a = ap.parse_args()

TAG = f"[{a.arm}]"
def log(*m):
    print(TAG, *m, flush=True)

try:
    T0 = float(open("/tmp/pg_session_start").read())
except Exception:
    T0 = time.time()
dev = torch.device("cuda")
torch.backends.cudnn.benchmark = True
os.makedirs(a.ckpt_dir, exist_ok=True)

def save_atomic(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    with open(tmp, "rb") as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, path)

def minmax_pm1(x):                                   # per-image min-max -> [-1,1] (= original z-score + ScaleIntensity)
    mn = x.amin(dim=(1, 2, 3), keepdim=True)
    mx = x.amax(dim=(1, 2, 3), keepdim=True)
    return 2 * (x - mn) / (mx - mn + 1e-8) - 1

# ---------------- original HighFreqLoss (copied from monai_ddpm.py) ----------------
_sob = torch.tensor([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=torch.float32)
KX = (_sob[None, None] / 8).to(dev)
KY = (_sob.t()[None, None] / 8).to(dev)
def hf_loss(x, y):
    gx = F.conv2d(x, KX, padding=1) - F.conv2d(y, KX, padding=1)
    gy = F.conv2d(x, KY, padding=1) - F.conv2d(y, KY, padding=1)
    return (gx.pow(2) + gy.pow(2)).mean()

# ---------------- data (uint8 cache, kept on GPU) ----------------
d = np.load("/tmp/train_cache.npz")
X = torch.from_numpy(d["x"])
Y = torch.from_numpy(d["y"]).long()
_rng = np.random.default_rng(123)                          # same subset for both arms, every run
_keep = []
for _c in (0, 1):
    _idx = np.where(d["y"] == _c)[0]
    _keep += sorted(_rng.choice(_idx, int(round(a.data_frac * len(_idx))), replace=False).tolist())
_keep = np.array(sorted(_keep))
X, Y = X[_keep], Y[_keep]
N = X.shape[0]
Xg, Yg = X.to(dev), Y.to(dev)
log(f"data (frac={a.data_frac}) {N} images {tuple(X.shape[1:])} benign={int((Y == 0).sum())} malignant={int((Y == 1).sum())} "
    f"on {torch.cuda.get_device_name(0)}")

def augment(img, gg):
    # img: (B,1,H,W) float in 0..255. Effective original aug: shear +-0.1, scale +-5% (p=0.9), h-flip p=0.5
    B = img.shape[0]
    do = (torch.rand(B, generator=gg, device=dev) < 0.9).float()
    u = lambda: (torch.rand(B, generator=gg, device=dev) * 2 - 1)
    sx, sy, shx, shy = 1 + 0.05 * u() * do, 1 + 0.05 * u() * do, 0.1 * u() * do, 0.1 * u() * do
    zero = torch.zeros(B, device=dev)
    theta = torch.stack([torch.stack([sx, shx, zero], 1), torch.stack([shy, sy, zero], 1)], 1)
    grid = F.affine_grid(theta, list(img.shape), align_corners=False)
    out = F.grid_sample(img, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    flip = torch.rand(B, generator=gg, device=dev) < 0.5
    return torch.where(flip.view(-1, 1, 1, 1), out.flip(-1), out)

# ---------------- model: the repo's exact DiffusionModelUNet + noise schedule ----------------
from model_architectures import MonaiDDPM
torch.manual_seed(a.seed)                       # identical initial weights for both arms
_m = MonaiDDPM(lr=a.lr, T=1000)
unet = _m.unet.to(dev)
ac = _m.scheduler.alphas_cumprod.detach().float().to(dev)
del _m
T = ac.numel()
opt = torch.optim.Adam(unet.parameters(), lr=a.lr)
scaler = torch.amp.GradScaler("cuda")
nlf = PhysicsNLF(dev)
tau_lo = tau_hi = None

# ---------------- resume ----------------
paths = [os.path.join(a.ckpt_dir, f"{a.arm}_last.pt")]
if not a.smoke_batches:
    paths += glob.glob(f"/kaggle/input/**/{a.arm}_last.pt", recursive=True)
cands = []
for p in paths:
    if os.path.exists(p):
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
            cands.append((int(ck["step"]), p))
            del ck
        except Exception as e:
            log("unreadable checkpoint (skipped):", p, repr(e)[:150])
start_epoch, step, opt_steps, history, resumed = 0, 0, 0, [], None
if cands:
    resumed = max(cands)[1]
    ck = torch.load(resumed, map_location="cpu", weights_only=False)
    for k in ("lam", "lr", "bs", "accum", "seed", "p_uncond", "data_frac", "band_lo", "band_hi"):
        if ck["args"].get(k) != getattr(a, k):
            log(f"!! WARNING {k} differs: checkpoint={ck['args'].get(k)} now={getattr(a, k)}")
    unet.load_state_dict(ck["unet"])
    opt.load_state_dict(ck["opt"])
    scaler.load_state_dict(ck["scaler"])
    start_epoch, step, opt_steps, history = int(ck["epoch"]), int(ck["step"]), int(ck["opt_steps"]), ck["history"]
    nlf.load(ck["nlf"])
    tau_lo, tau_hi = float(ck["tau_lo"]), float(ck["tau_hi"])
    del ck
    torch.cuda.empty_cache()

# ---------------- physics calibration on REAL images, in the training intensity domain ----------------
Xf = X.float()
mn = Xf.amin(dim=(1, 2), keepdim=True); mx = Xf.amax(dim=(1, 2), keepdim=True)
Xmm = ((Xf - mn) / (mx - mn + 1e-8) * 255).round().clamp(0, 255).to(torch.uint8)
del Xf
if nlf.p is None:
    rep = nlf.fit(Xmm)
    log(f"NLF fit: sigma^2 = {rep['b']:.3e} + {rep['a']:.3e}*mu + {rep['c']:.3e}*mu^2 | "
        f"R2(var)={rep['R2_var']:.3f} R2(logvar)={rep['R2_logvar']:.3f}")
with torch.no_grad():
    res_real = torch.cat([nlf.residual(Xmm[i:i + 32].to(dev).float().div(255).unsqueeze(1)).cpu()
                          for i in range(0, N, 32)])
if tau_lo is None:
    tau_lo = float(torch.quantile(res_real, a.band_lo))
    tau_hi = float(torch.quantile(res_real, a.band_hi))
    with open(os.path.join(a.ckpt_dir, f"nlf_fit_{a.arm}.json"), "w") as fh:
        json.dump({**rep, "tau_lo": tau_lo, "tau_hi": tau_hi, "band": [a.band_lo, a.band_hi],
                   "real_mean_residual": float(res_real.mean())}, fh, indent=2)
log(f"physics residual of REAL images: mean={float(res_real.mean()):.4f} | "
    f"band [q{a.band_lo}, q{a.band_hi}] = [{tau_lo:.4f}, {tau_hi:.4f}]")

n_opt = sum(1 for s in opt.state.values() if s)
chk = float(sum(p.detach().float().sum() for p in list(unet.parameters())[:5]))
log(f"START epoch={start_epoch} step={step} opt_states={n_opt} lam={a.lam} checksum={chk:.6f} "
    f"resumed={resumed or 'scratch'}")
if resumed and n_opt == 0:
    log("!! WARNING: checkpoint loaded but optimizer state is EMPTY -> not a true resume")

spe = N // a.bs
if a.smoke_batches:
    spe = min(spe, a.smoke_batches)
durs = [h["minutes"] * 60 for h in history[-3:]]
reason = "max epochs reached"
bad = 0
for epoch in range(start_epoch, a.epochs):
    used_h = (time.time() - T0) / 3600
    if durs and used_h + 1.15 * max(durs[-3:]) / 3600 + 0.2 > a.budget_h:
        reason = f"time budget ({used_h:.2f} h used of {a.budget_h} h)"
        if epoch > start_epoch and epoch % a.ckpt_every != 0:   # make sure the newest epoch is resumable
            save_atomic({"unet": unet.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                         "epoch": epoch, "step": step, "opt_steps": opt_steps, "args": vars(a), "nlf": nlf.state(),
                         "tau_lo": tau_lo, "tau_hi": tau_hi, "history": history},
                        os.path.join(a.ckpt_dir, f"{a.arm}_last.pt"))
            log(f"budget stop: saved resumable checkpoint at epoch {epoch}")
        break
    gcpu = torch.Generator().manual_seed(a.seed * 1000 + epoch)
    perm = torch.randperm(N, generator=gcpu)
    gg = torch.Generator(device=dev).manual_seed(a.seed * 1000 + epoch)
    unet.train()
    opt.zero_grad(set_to_none=True)
    t_ep, t_win = time.time(), time.time()
    S = dict(loss=0.0, comp=0.0, mse=0.0, ms=0.0, hf=0.0, phys=0.0, hinge=0.0)
    n_ok = 0
    for i in range(spe):
        idx = perm[i * a.bs:(i + 1) * a.bs].to(dev)
        x0 = minmax_pm1(augment(Xg[idx].float().unsqueeze(1), gg))
        y = Yg[idx]
        t = torch.randint(0, T, (a.bs,), generator=gg, device=dev)
        eps = torch.randn(x0.shape, generator=gg, device=dev)
        drop = torch.rand(a.bs, generator=gg, device=dev) < a.p_uncond
        cond = torch.where(drop, torch.full_like(y, -1), y).float().view(a.bs, 1, 1)
        ab = ac[t].view(-1, 1, 1, 1)
        xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
        with torch.autocast("cuda", dtype=torch.float16):
            eps_hat = unet(xt, timesteps=t, context=cond)
        eps_hat = eps_hat.float()
        mse = F.mse_loss(eps_hat, eps)
        x0_hat = ((xt - (1 - ab).sqrt() * eps_hat) / ab.sqrt()).clamp(-1, 1)
        ms = 1.0 - msssim((x0_hat + 1) / 2, (x0 + 1) / 2, data_range=2.0, gaussian_kernel=False, normalize="relu")
        hf = hf_loss(x0_hat, x0)
        comp = 0.7 * mse + 0.2 * ms + 0.05 * hf                     # exact original loss
        w = ab.view(-1)
        if a.lam > 0:
            r = nlf.residual((x0_hat + 1) / 2)
            hinge = (w * (F.relu(r - tau_hi) + F.relu(tau_lo - r))).mean()
            loss = comp + a.lam * hinge
        else:
            with torch.no_grad():                                   # baseline: tracked, never trained on
                r = nlf.residual((x0_hat + 1) / 2)
                hinge = (w * (F.relu(r - tau_hi) + F.relu(tau_lo - r))).mean()
            loss = comp
        phys = (w * r).mean().detach()
        if not torch.isfinite(loss):
            bad += 1
            if bad > 50:
                raise RuntimeError("50 consecutive non-finite losses - stopping")
            continue
        bad = 0
        scaler.scale(loss / a.accum).backward()
        step += 1
        if (i + 1) % a.accum == 0 or i == spe - 1:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            opt_steps += 1
        n_ok += 1
        for k, v in (("loss", loss), ("comp", comp), ("mse", mse), ("ms", ms), ("hf", hf), ("phys", phys), ("hinge", hinge)):
            S[k] += float(v.detach())
        if (i + 1) % a.log_every == 0:
            ips = a.log_every / (time.time() - t_win)
            t_win = time.time()
            m = {k: v / n_ok for k, v in S.items()}
            log(f"ep {epoch + 1} it {i + 1}/{spe} step {step} loss {m['loss']:.4f} comp {m['comp']:.4f} "
                f"mse {m['mse']:.4f} ms {m['ms']:.4f} hf {m['hf']:.4f} phys {m['phys']:.4f} "
                f"hinge {m['hinge']:.4f} | {ips:.2f} it/s")
    dur = time.time() - t_ep
    durs.append(dur)
    m = {k: v / max(n_ok, 1) for k, v in S.items()}
    history.append({"epoch": epoch + 1, "step": step, "opt_steps": opt_steps, **m, "minutes": dur / 60})
    if (epoch + 1) % a.ckpt_every == 0 or epoch + 1 == a.epochs:
      save_atomic({"unet": unet.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                 "epoch": epoch + 1, "step": step, "opt_steps": opt_steps, "args": vars(a), "nlf": nlf.state(),
                 "tau_lo": tau_lo, "tau_hi": tau_hi, "history": history}, os.path.join(a.ckpt_dir, f"{a.arm}_last.pt"))
    snap = {"unet": {k: v.detach().half().cpu() for k, v in unet.state_dict().items()},
            "epoch": epoch + 1, "step": step, "arm": a.arm, "lam": a.lam, "nlf": nlf.state(), "tau_lo": tau_lo, "tau_hi": tau_hi}
    save_atomic(snap, os.path.join(a.ckpt_dir, f"{a.arm}_w_latest.pt"))
    if (epoch + 1) % a.snap_every == 0:
        save_atomic(snap, os.path.join(a.ckpt_dir, f"{a.arm}_w_ep{epoch + 1:03d}.pt"))
    del snap
    with open(os.path.join(a.ckpt_dir, f"{a.arm}_history.json"), "w") as fh:
        json.dump(history, fh, indent=1)
    left = a.epochs - epoch - 1
    log(f"SAVED epoch={epoch + 1} step={step} opt_steps={opt_steps} loss={m['loss']:.4f} comp={m['comp']:.4f} "
        f"mse={m['mse']:.4f} ms={m['ms']:.4f} hf={m['hf']:.4f} phys={m['phys']:.4f} hinge={m['hinge']:.4f} | "
        f"{dur / 60:.1f} min/epoch | peak GPU mem {torch.cuda.max_memory_allocated() / 1e9:.1f} GB | "
        f"~{left * dur / 3600:.1f} h for {left} more")
done_ep = history[-1]["epoch"] if history else start_epoch
if a.final and done_ep >= a.epochs:
    with open(os.path.join(a.ckpt_dir, f"{a.arm}_done.json"), "w") as fh:
        json.dump({"arm": a.arm, "epochs": done_ep, "step": step, "opt_steps": opt_steps, "args": vars(a)}, fh, indent=2)
    for f in (f"{a.arm}_last.pt", f"{a.arm}_w_latest.pt"):
        p = os.path.join(a.ckpt_dir, f)
        if os.path.exists(p):
            os.remove(p)
    log("final: wrote done marker, removed resume files (snapshots kept)")
log(f"FINISHED epochs_completed={done_ep} step={step} reason={reason}")
''')
Path("eval_arm3.py").write_text(r'''# eval_arm3.py - sample one arm (DDIM + classifier-free guidance) and score it.
# Adds to eval_arm.py: mammography-domain metrics (KID and precision/recall in the oracle's feature space),
# Inception precision/recall, per-sample physics residuals, and saves the samples for the downstream test.
# Same seeds for both arms -> sample i of baseline and sample i of pinn start from the SAME noise (paired test).
import os, sys, glob, json, time, argparse
import numpy as np
import torch

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
sys.path.insert(0, WD)
from pinn_physics import PhysicsNLF

ap = argparse.ArgumentParser()
ap.add_argument("--arm", required=True)
ap.add_argument("--epoch", type=int, default=0)          # 0 = latest snapshot
ap.add_argument("--n_per_class", type=int, default=200)
ap.add_argument("--steps", type=int, default=50)
ap.add_argument("--gs", type=float, default=3.0)
ap.add_argument("--bs", type=int, default=16)
ap.add_argument("--seed", type=int, default=2025)
ap.add_argument("--ckpt_dir", default="/kaggle/working/ckpt")
ap.add_argument("--out", default="/kaggle/working/eval")
a = ap.parse_args()
TAG = f"[eval-{a.arm}]"
def log(*m):
    print(TAG, *m, flush=True)
dev = torch.device("cuda")
os.makedirs(a.out, exist_ok=True)

name = f"{a.arm}_w_latest.pt" if a.epoch == 0 else f"{a.arm}_w_ep{a.epoch:03d}.pt"
found = glob.glob(os.path.join(a.ckpt_dir, name)) + glob.glob(f"/kaggle/input/**/{name}", recursive=True)
if not found:
    raise SystemExit(f"{TAG} snapshot {name} not found in {a.ckpt_dir} or /kaggle/input")
snap = torch.load(found[0], map_location="cpu", weights_only=False)
log(f"snapshot {found[0]} | epoch={snap['epoch']} step={snap['step']} lam={snap['lam']}")

from model_architectures import MonaiDDPM
_m = MonaiDDPM(lr=1e-5, T=1000)
unet = _m.unet
ac = _m.scheduler.alphas_cumprod.detach().float().to(dev)
del _m
unet.load_state_dict({k: v.float() for k, v in snap["unet"].items()})
unet = unet.half().to(dev).eval()
T = ac.numel()
nlf = PhysicsNLF(dev)
nlf.load(snap["nlf"])

@torch.no_grad()
def ddim(label, n, gen):
    x = torch.randn(n, 1, 256, 256, generator=gen, device=dev)
    ts = torch.linspace(T - 1, 0, a.steps, device=dev).round().long()
    cond = torch.full((n, 1, 1), float(label), device=dev)
    ctx = torch.cat([-torch.ones_like(cond), cond]).half()
    for i, t in enumerate(ts):
        ab_t = ac[t]
        ab_prev = ac[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0, device=dev)
        tb = torch.full((2 * n,), int(t), device=dev, dtype=torch.long)
        out = unet(torch.cat([x, x]).half(), timesteps=tb, context=ctx).float()
        eu, ec = out.chunk(2)
        eps = eu + a.gs * (ec - eu)
        x0 = ((x - (1 - ab_t).sqrt() * eps) / ab_t.sqrt()).clamp(-1, 1)
        eps = (x - ab_t.sqrt() * x0) / (1 - ab_t).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
    return ((x + 1) / 2).clamp(0, 1)

# ---------------- generate ----------------
t0 = time.time()
imgs, labels = [], []
for label in (0, 1):
    done, b = 0, 0
    while done < a.n_per_class:
        n = min(a.bs, a.n_per_class - done)
        gen = torch.Generator(device=dev).manual_seed(a.seed + 100000 * label + b)
        imgs.append(ddim(label, n, gen).cpu())
        labels += [label] * n
        done += n
        b += 1
    log(f"class {label}: {done} samples ({(time.time() - t0) / 60:.1f} min)")
G = torch.cat(imgs)                                    # (M,1,256,256) in [0,1]
L = torch.tensor(labels)

# ---------------- real test images ----------------
d = np.load("/tmp/test_cache.npz")
R = torch.from_numpy(d["x"]).float().div(255).unsqueeze(1)
RL = torch.from_numpy(d["y"]).long()

# ---------------- oracle classifier ----------------
root = "/kaggle/working/U24_synthesis"
sys.path.insert(0, root)
cwd = os.getcwd(); os.chdir(root)
from models import get_model
os.chdir(cwd)
op = glob.glob("/kaggle/input/**/oracle-checkpoint-final/last.pt", recursive=True)
res = {"arm": a.arm, "epoch": snap["epoch"], "step": snap["step"], "lam": snap["lam"],
       "n_per_class": a.n_per_class, "ddim_steps": a.steps, "guidance": a.gs}
correct = np.zeros(len(L), dtype=bool)
if op:
    orc = get_model("convnext_tiny", pretrained=False)
    orc.load_state_dict(torch.load(op[0], map_location="cpu", weights_only=False)["model_state"])
    orc = orc.half().to(dev).eval()
    last_lin = [m for m in orc.modules() if isinstance(m, torch.nn.Linear)][-1]
    _feat = {}
    last_lin.register_forward_hook(lambda mod, inp, out: _feat.__setitem__("f", inp[0].detach().float().flatten(1).cpu()))
    FEAT = {}
    def oracle_pred(I01, key):
        out, feats = [], []
        for i in range(0, len(I01), 32):
            x = I01[i:i + 32].to(dev)
            mn = x.amin(dim=(1, 2, 3), keepdim=True); mx = x.amax(dim=(1, 2, 3), keepdim=True)
            x = ((x - mn) / (mx - mn + 1e-8) - 0.5) / 0.5
            with torch.no_grad():
                out.append(orc(x.half()).argmax(1).cpu())
            feats.append(_feat["f"])
        FEAT[key] = torch.cat(feats)
        return torch.cat(out)
    pr = oracle_pred(R, "real")
    res["oracle_real_acc"] = {c: float((pr[RL == k] == k).float().mean()) for k, c in enumerate(["benign", "malignant"])}
    pg = oracle_pred(G, "gen")
    correct = (pg == L).numpy()
    res["oracle_acc"] = {c: float(correct[L.numpy() == k].mean()) for k, c in enumerate(["benign", "malignant"])}
    res["oracle_acc"]["mean"] = float(correct.mean())
    log(f"oracle on REAL test: {res['oracle_real_acc']} | on generated: {res['oracle_acc']}")
else:
    log("oracle checkpoint not found (attach oracle-checkpoint-final) - skipping oracle accuracy")

# ---------------- physics residual (generated vs real) ----------------
def all_res(I):
    return torch.cat([nlf.residual(I[i:i + 32].to(dev)).cpu() for i in range(0, len(I), 32)])
rG, rR = all_res(G), all_res(R)
res["physics_residual_generated"] = float(rG.mean())
res["physics_residual_real_test"] = float(rR.mean())
res["physics_abs_gap_to_real"] = abs(float(rG.mean()) - float(rR.mean()))
res["physics_residual_generated_std"] = float(rG.std())
res["physics_residual_real_std"] = float(rR.std())
log(f"physics residual: generated={res['physics_residual_generated']:.4f} real={res['physics_residual_real_test']:.4f}")

# ---------------- feature-space metrics (KID with polynomial kernel, k-NN precision/recall) ----------------
def kid_feats(fr, ff, n_sub=100, sub=100, seed=0):
    fr, ff = fr.double(), ff.double()
    d = fr.shape[1]; m = min(sub, len(fr), len(ff))
    g = torch.Generator().manual_seed(seed); vals = []
    for _ in range(n_sub):
        x = fr[torch.randperm(len(fr), generator=g)[:m]]; y = ff[torch.randperm(len(ff), generator=g)[:m]]
        kxx = (x @ x.T / d + 1) ** 3; kyy = (y @ y.T / d + 1) ** 3; kxy = (x @ y.T / d + 1) ** 3
        mmd = ((kxx.sum() - kxx.diag().sum()) / (m * (m - 1)) + (kyy.sum() - kyy.diag().sum()) / (m * (m - 1))
               - 2 * kxy.mean())
        vals.append(float(mmd))
    return float(np.mean(vals)), float(np.std(vals))

def prec_rec(fr, ff, k=3):
    fr, ff = fr.float(), ff.float()
    rr = torch.cdist(fr, fr).kthvalue(k + 1, dim=1).values        # k-th neighbour radius (self is 0)
    rf = torch.cdist(ff, ff).kthvalue(k + 1, dim=1).values
    dfr = torch.cdist(ff, fr)
    precision = float((dfr <= rr[None, :]).any(1).float().mean())   # generated samples inside real manifold
    recall = float((dfr.T <= rf[None, :]).any(1).float().mean())    # real samples covered by generated manifold
    return precision, recall

if op:
    res["KID_mammo_mean"], res["KID_mammo_std"] = kid_feats(FEAT["real"], FEAT["gen"])
    res["precision_mammo"], res["recall_mammo"] = prec_rec(FEAT["real"], FEAT["gen"])
    log(f"mammography-space KID = {res['KID_mammo_mean']:.4f} +- {res['KID_mammo_std']:.4f} | "
        f"precision {res['precision_mammo']:.3f} recall {res['recall_mammo']:.3f}")

# ---------------- KID / FID ----------------
def rgb8(I):
    return (I.clamp(0, 1) * 255).round().to(torch.uint8).repeat(1, 3, 1, 1)
try:
    from torchmetrics.image.kid import KernelInceptionDistance
    kid = KernelInceptionDistance(subset_size=min(100, len(G), len(R))).to(dev)
    for i in range(0, len(R), 32):
        kid.update(rgb8(R[i:i + 32]).to(dev), real=True)
    for i in range(0, len(G), 32):
        kid.update(rgb8(G[i:i + 32]).to(dev), real=False)
    km, ks = kid.compute()
    res["KID_mean"], res["KID_std"] = float(km), float(ks)
    log(f"KID = {float(km):.4f} +- {float(ks):.4f}")
    try:
        fr_i = torch.cat([f.cpu() for f in kid.real_features]); ff_i = torch.cat([f.cpu() for f in kid.fake_features])
        res["precision_inception"], res["recall_inception"] = prec_rec(fr_i, ff_i)
        log(f"Inception precision {res['precision_inception']:.3f} recall {res['recall_inception']:.3f}")
    except Exception as e:
        log("Inception precision/recall skipped:", repr(e)[:150])
except Exception as e:
    log("KID skipped:", repr(e)[:200])
try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    fid = FrechetInceptionDistance(feature=2048).to(dev)
    for i in range(0, len(R), 32):
        fid.update(rgb8(R[i:i + 32]).to(dev), real=True)
    for i in range(0, len(G), 32):
        fid.update(rgb8(G[i:i + 32]).to(dev), real=False)
    res["FID"] = float(fid.compute())
    log(f"FID = {res['FID']:.2f}")
except Exception as e:
    log("FID skipped:", repr(e)[:200])

# ---------------- save ----------------
from PIL import Image
tag = f"{a.arm}_ep{snap['epoch']:03d}"
k = min(8, a.n_per_class)
rows = [np.concatenate([(G[j + c * a.n_per_class, 0].numpy() * 255).astype(np.uint8) for j in range(k)], axis=1)
        for c in (0, 1)]
Image.fromarray(np.concatenate(rows, axis=0)).save(os.path.join(a.out, f"grid_{tag}.png"))
np.savez_compressed(os.path.join(a.out, f"samples_{tag}.npz"),
                    x=(G[:, 0].numpy() * 255).astype(np.uint8), y=L.numpy(), correct=correct,
                    phys=rG.numpy())
with open(os.path.join(a.out, f"results_{tag}.json"), "w") as fh:
    json.dump(res, fh, indent=2)
log(f"EVAL DONE -> {a.out}/results_{tag}.json ({(time.time() - t0) / 60:.1f} min)")
''')
Path("classify.py").write_text(r'''# classify.py - DOWNSTREAM UTILITY TEST. Train a benign/malignant classifier (ResNet-18) on
#   (a) the same 25% REAL subset the generators saw, optionally (b) + synthetic images from one generator,
# then test on the untouched REAL CBIS-DDSM test set (AUC, balanced accuracy).
import os, sys, json, time, argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

ap = argparse.ArgumentParser()
ap.add_argument("--synth", default="")                   # path to samples_<arm>_epXXX.npz ("" = real only)
ap.add_argument("--tag", required=True)                  # e.g. real_only | base3_s42 | pinn3_s42
ap.add_argument("--clf_seed", type=int, default=0)
ap.add_argument("--data_frac", type=float, default=0.25)
ap.add_argument("--epochs", type=int, default=20)
ap.add_argument("--bs", type=int, default=32)
ap.add_argument("--lr", type=float, default=1e-4)
ap.add_argument("--max_batches", type=int, default=0)    # smoke test only
ap.add_argument("--out", default="/kaggle/working/downstream")
a = ap.parse_args()
dev = torch.device("cuda")
os.makedirs(a.out, exist_ok=True)
TAG = f"[clf-{a.tag}-c{a.clf_seed}]"
def log(*m):
    print(TAG, *m, flush=True)

# same stratified subset as train_arm3.py (rng 123)
d = np.load("/tmp/train_cache.npz")
rng = np.random.default_rng(123); keep = []
for c in (0, 1):
    idx = np.where(d["y"] == c)[0]
    keep += sorted(rng.choice(idx, int(round(a.data_frac * len(idx))), replace=False).tolist())
keep = np.array(sorted(keep))
Xtr, Ytr = d["x"][keep], d["y"][keep]
n_real = len(Ytr)
if a.synth:
    s = np.load(a.synth)
    Xtr = np.concatenate([Xtr, s["x"]]); Ytr = np.concatenate([Ytr, s["y"]])
t = np.load("/tmp/test_cache.npz")
Xte, Yte = t["x"], t["y"]
log(f"train: {n_real} real + {len(Ytr) - n_real} synthetic | test: {len(Yte)} real")

def prep(xu8):                                           # per-image min-max -> [-1,1], 3 channels
    x = torch.from_numpy(xu8).float().unsqueeze(1)
    mn = x.amin(dim=(1, 2, 3), keepdim=True); mx = x.amax(dim=(1, 2, 3), keepdim=True)
    x = ((x - mn) / (mx - mn + 1e-8) - 0.5) / 0.5
    return x.repeat(1, 3, 1, 1)

torch.manual_seed(a.clf_seed); np.random.seed(a.clf_seed)
import torchvision
try:
    net = torchvision.models.resnet18(weights="IMAGENET1K_V1")
    log("ImageNet-pretrained ResNet-18")
except Exception as e:
    net = torchvision.models.resnet18(weights=None)
    log("pretrained weights unavailable -> random init:", repr(e)[:100])
net.fc = nn.Linear(net.fc.in_features, 2)
net = net.to(dev)
opt = torch.optim.AdamW(net.parameters(), lr=a.lr, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda")
Xt, Yt = prep(Xtr), torch.from_numpy(Ytr).long()
g = torch.Generator().manual_seed(a.clf_seed)
t0 = time.time()
for ep in range(a.epochs):
    net.train()
    perm = torch.randperm(len(Yt), generator=g)
    nb = 0
    for i in range(0, len(perm), a.bs):
        idx = perm[i:i + a.bs]
        x, y = Xt[idx].to(dev), Yt[idx].to(dev)
        flip = torch.rand(len(idx), generator=g) < 0.5
        x = torch.where(flip.to(dev).view(-1, 1, 1, 1), x.flip(-1), x)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = F.cross_entropy(net(x), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        nb += 1
        if a.max_batches and nb >= a.max_batches:
            break

net.eval(); probs = []
Xe = prep(Xte)
with torch.no_grad():
    for i in range(0, len(Xe), 64):
        with torch.autocast("cuda", dtype=torch.float16):
            probs.append(torch.softmax(net(Xe[i:i + 64].to(dev)).float(), 1)[:, 1].cpu())
p = torch.cat(probs).numpy()
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
auc = float(roc_auc_score(Yte, p)); bacc = float(balanced_accuracy_score(Yte, (p > 0.5).astype(int)))
res = {"tag": a.tag, "clf_seed": a.clf_seed, "synth": a.synth, "n_real": int(n_real),
       "n_synth": int(len(Ytr) - n_real), "epochs": a.epochs, "auc": auc, "balanced_acc": bacc}
with open(os.path.join(a.out, f"clf_{a.tag}_c{a.clf_seed}.json"), "w") as fh:
    json.dump(res, fh, indent=2)
log(f"DONE AUC={auc:.4f} balanced_acc={bacc:.4f} ({(time.time() - t0) / 60:.1f} min)")
''')
print("wrote pinn_physics.py, train_arm3.py, eval_arm3.py, classify.py")

print(subprocess.run("nvidia-smi --query-gpu=index,name,memory.used,memory.total --format=csv",
                     shell=True, capture_output=True, text=True).stdout)
print(subprocess.run([sys.executable, "-c", "import torch; print('torch', torch.__version__, '| GPUs visible:', torch.cuda.device_count())"],
                     capture_output=True, text=True).stdout)

def run_parallel(jobs, poll_min=20):
    """jobs = [(gpu_index, [script, args...]), ...] -> runs them at the same time, one per GPU."""
    procs, outs = [], []
    for gpu, argv in jobs:
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
        env["PYTHONUNBUFFERED"] = "1"
        p = subprocess.Popen([sys.executable, "-u"] + argv, cwd=WD, env=env, text=True, bufsize=1,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        procs.append(p); outs.append([])
    def pump(p, buf, gpu):
        for line in p.stdout:
            buf.append(line)
            print(line if line.startswith("[") else f"[gpu{gpu}] {line}", end="", flush=True)
    def poll():
        while any(p.poll() is None for p in procs):
            time.sleep(poll_min * 60)
            if any(p.poll() is None for p in procs):
                q = subprocess.run("nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader",
                                   shell=True, capture_output=True, text=True).stdout.strip().replace("\n", " | ")
                print(f"[nvidia-smi] {q}", flush=True)
    ths = [threading.Thread(target=pump, args=(p, b, g)) for p, b, (g, _) in zip(procs, outs, jobs)]
    for th in ths: th.start()
    threading.Thread(target=poll, daemon=True).start()
    for th in ths: th.join()
    return [p.wait() for p in procs], ["".join(b) for b in outs]
print("Cell 2 done.")


In [ ]:
# ===================== CELL 4 — SMOKE TEST (train mode only, ~12 min) =====================
# Tests training, resume, atomic saving, evaluation (all new metrics) and the downstream classifier.
import os, re, glob, shutil, json
SMOKE_OK = False
if MODE != "train":
    print("eval mode -> smoke test skipped")
else:
    SM, EV, DS = "/kaggle/working/ckpt_smoke", "/kaggle/working/eval_smoke", "/kaggle/working/downstream_smoke"
    for p in (SM, EV, DS): shutil.rmtree(p, ignore_errors=True)
    common = ["--data_frac", str(DATA_FRAC), "--smoke_batches", "30", "--ckpt_dir", SM, "--snap_every", "1", "--log_every", "10"]
    def train_jobs(ep):
        return [(0, ["train_arm3.py", "--arm", "base3_sm", "--lam", "0.0", "--epochs", str(ep)] + common),
                (1, ["train_arm3.py", "--arm", "pinn3_sm", "--lam", str(LAM), "--epochs", str(ep)] + common)]
    ev_args = ["--epoch", "2", "--n_per_class", "8", "--steps", "5", "--bs", "8", "--ckpt_dir", SM, "--out", EV]
    rc1, o1 = run_parallel(train_jobs(1), poll_min=5)
    rc2, o2 = run_parallel(train_jobs(2), poll_min=5) if rc1 == [0, 0] else ([1, 1], ["", ""])
    rc3, o3 = (run_parallel([(0, ["eval_arm3.py", "--arm", "base3_sm"] + ev_args),
                             (1, ["eval_arm3.py", "--arm", "pinn3_sm"] + ev_args)], poll_min=5)
               if rc2 == [0, 0] else ([1, 1], ["", ""]))
    clf_args = ["--data_frac", str(DATA_FRAC), "--epochs", "1", "--max_batches", "3", "--out", DS]
    rc4, o4 = (run_parallel([(0, ["classify.py", "--tag", "real_only"] + clf_args),
                             (1, ["classify.py", "--tag", "pinn3_sm", "--synth", f"{EV}/samples_pinn3_sm_ep002.npz"] + clf_args)],
                            poll_min=5) if rc3 == [0, 0] else ([1, 1], ["", ""]))
    S = r"START epoch=(\d+) step=(\d+) opt_states=(\d+) lam=\S+ checksum=(\S+)"
    def g(pat, s):
        m = re.search(pat, s); return m.groups() if m else None
    def lc(s):
        m = re.findall(r"loss (\S+) comp (\S+) mse", s); return (float(m[-1][0]), float(m[-1][1])) if m else None
    def rj(p):
        return json.load(open(p))
    checks = []
    for k, arm in enumerate(["base3_sm", "pinn3_sm"]):
        checks += [
            (f"{arm}: run 1 started from scratch",              lambda k=k: g(S, o1[k])[:2] == ("0", "0")),
            (f"{arm}: run 2 RESUMED at epoch 1, step 30",      lambda k=k: g(S, o2[k])[:2] == ("1", "30")),
            (f"{arm}: run 2 optimizer state restored",         lambda k=k: int(g(S, o2[k])[2]) > 0),
            (f"{arm}: files written atomically",
             lambda arm=arm: all(os.path.exists(f"{SM}/{arm}_{x}") for x in ["last.pt", "w_ep001.pt", "w_ep002.pt", "history.json"])
                             and not glob.glob(f"{SM}/*.tmp")),
            (f"{arm}: evaluation incl. mammography metrics",   lambda arm=arm: "KID_mammo_mean" in rj(f"{EV}/results_{arm}_ep002.json")),
        ]
    checks += [
        ("both arms start from IDENTICAL weights",            lambda: g(S, o1[0])[3] == g(S, o1[1])[3]),
        ("base3 loss == original composite loss",            lambda: abs(lc(o2[0])[0] - lc(o2[0])[1]) < 1e-6),
        ("pinn3 loss > composite (physics band active)",     lambda: lc(o2[1])[0] > lc(o2[1])[1]),
        ("downstream classifier: real only",                  lambda: "auc" in rj(f"{DS}/clf_real_only_c0.json")),
        ("downstream classifier: real + synthetic",           lambda: rj(f"{DS}/clf_pinn3_sm_c0.json")["n_synth"] == 16),
    ]
    def ok(f):
        try: return bool(f())
        except Exception: return False
    print("\n" + "=" * 64)
    for name, f in checks:
        print(("PASS  " if ok(f) else "FAIL  ") + name)
    SMOKE_OK = all(ok(f) for _, f in checks)
    if SMOKE_OK:
        for p in (SM, EV, DS): shutil.rmtree(p, ignore_errors=True)
        print("ALL PASS -> smoke files deleted, Cell 5 will train.")
    else:
        print("SMOKE TEST FAILED -> Cell 5 will not train. Send Claude this block + the first error lines above.")


In [ ]:
# ===================== CELL 5 — TRAINING (train mode): 3 seeds x (base3 on GPU0 + pinn3 on GPU1) =====================
# Each pair ~2.3 h. Resumable: if the session ends, attach this version's output next time and run again with
# MODE="train" - finished runs are skipped (done markers), unfinished runs continue from their checkpoint.
import glob, time, subprocess
def is_done(arm):
    return bool(glob.glob(f"/kaggle/working/ckpt/{arm}_done.json") + glob.glob(f"/kaggle/input/**/{arm}_done.json", recursive=True))
if MODE != "train":
    print("eval mode -> training skipped")
elif not globals().get("SMOKE_OK", False):
    print("Smoke test did not pass -> NOT training.")
else:
    PAIR_H = 2.6                                          # conservative hours per pair
    for s in SEEDS:
        arms = [(0, f"base3_s{s}", 0.0), (1, f"pinn3_s{s}", LAM)]
        todo = [x for x in arms if not is_done(x[1])]
        if not todo:
            print(f"seed {s}: both arms already done -> skip"); continue
        used = (time.time() - float(open("/tmp/pg_session_start").read())) / 3600
        if used + 0.5 > BUDGET_H:
            print(f"seed {s}: no time left ({used:.1f} h used of {BUDGET_H}) -> resume next session"); break
        jobs = [(gpu, ["train_arm3.py", "--arm", arm, "--lam", str(lam), "--seed", str(s), "--epochs", str(EPOCHS),
                       "--data_frac", str(DATA_FRAC), "--budget_h", str(BUDGET_H), "--snap_every", "15",
                       "--ckpt_every", "5", "--final"]) for gpu, arm, lam in todo]
        print(f"\n########## seed {s}: training {[a for _, a, _ in todo]} ##########")
        rc, out = run_parallel(jobs, poll_min=30)
        print(f"seed {s} exit codes:", rc)
    print("\nDone markers:", {f"{p}3_s{s}": is_done(f"{p}3_s{s}") for s in SEEDS for p in ("base", "pinn")})
    print(subprocess.run("ls -lh /kaggle/working/ckpt", shell=True, capture_output=True, text=True).stdout)
    if all(is_done(f"{p}3_s{s}") for s in SEEDS for p in ("base", "pinn")):
        print("ALL 6 RUNS FINISHED -> next: Session B with MODE='eval' (attach this version's output).")
    else:
        print("NOT all runs finished -> next session: MODE='train' again, attach this version's output.")


In [ ]:
# ===================== CELL 6 — EVALUATION (eval mode): 6 models, 200 images/class each, same noise =====================
import os, glob, time
if MODE != "eval":
    print("train mode -> evaluation skipped")
else:
    D = "/kaggle/working/eval"
    for s in SEEDS:
        arms = [f"base3_s{s}", f"pinn3_s{s}"]
        todo = [a for a in arms if not os.path.exists(f"{D}/results_{a}_ep{EPOCHS:03d}.json")]
        if not todo:
            print(f"seed {s}: already evaluated"); continue
        used = (time.time() - float(open("/tmp/pg_session_start").read())) / 3600
        if used + 1.0 > 11.0:
            print("not enough session time left -> run eval again in a new session"); break
        print(f"\n########## evaluating {todo} at epoch {EPOCHS} ##########")
        rc, out = run_parallel([(gpu, ["eval_arm3.py", "--arm", a, "--epoch", str(EPOCHS)]) for gpu, a in enumerate(todo)],
                               poll_min=15)
        print("exit codes:", rc)
    print(sorted(os.listdir(D)) if os.path.isdir(D) else "no eval folder")


In [ ]:
# ===================== CELL 7 — DOWNSTREAM TEST (eval mode): does synthetic data help a real classifier? =====================
# 3 conditions: real only | real + base3 synthetic | real + pinn3 synthetic. Tested on the REAL test set.
import os, json
if MODE != "eval":
    print("train mode -> downstream skipped")
else:
    D, DS = "/kaggle/working/eval", "/kaggle/working/downstream"
    CLF_SEEDS = [0, 1, 2]
    jobs = []
    for c in CLF_SEEDS:
        jobs.append(("real_only", "", c))
    for s in SEEDS:
        for p in ("base3", "pinn3"):
            npz = f"{D}/samples_{p}_s{s}_ep{EPOCHS:03d}.npz"
            if os.path.exists(npz):
                for c in CLF_SEEDS:
                    jobs.append((f"{p}_s{s}", npz, c))
            else:
                print("missing samples (run Cell 6 first):", npz)
    jobs = [j for j in jobs if not os.path.exists(f"{DS}/clf_{j[0]}_c{j[2]}.json")]
    print(f"{len(jobs)} classifier runs to do")
    for i in range(0, len(jobs), 2):
        batch = jobs[i:i + 2]
        run_parallel([(gpu, ["classify.py", "--tag", tag, "--clf_seed", str(c), "--data_frac", str(DATA_FRAC)]
                       + (["--synth", npz] if npz else [])) for gpu, (tag, npz, c) in enumerate(batch)], poll_min=15)
    print(sorted(os.listdir(DS)) if os.path.isdir(DS) else "no downstream folder")


In [ ]:
# ===================== CELL 8 — FINAL STATISTICS + PRE-REGISTERED VERDICT (eval mode) =====================
import os, json, glob, math
import numpy as np
if MODE != "eval":
    print("train mode -> summary skipped")
else:
    from scipy import stats
    D, DS = "/kaggle/working/eval", "/kaggle/working/downstream"
    R = {p: [json.load(open(f"{D}/results_{p}_s{s}_ep{EPOCHS:03d}.json")) for s in SEEDS] for p in ("base3", "pinn3")}
    M = [("KID_mean", "KID Inception", False), ("FID", "FID Inception", False),
         ("KID_mammo_mean", "KID mammography-space", False), ("precision_mammo", "precision (mammo)", True),
         ("recall_mammo", "recall (mammo)", True), ("precision_inception", "precision (Inception)", True),
         ("recall_inception", "recall (Inception)", True), ("physics_abs_gap_to_real", "|physics - real|", False)]
    def get(r, k):
        if k.startswith("oracle"):
            return r["oracle_acc"]["mean"]
        return r.get(k, float("nan"))
    M.append(("oracle_mean", "oracle class accuracy", True))
    out = {"metrics": {}, "downstream": {}}
    print("=" * 108)
    print(f"{'metric':26s}{'base3 mean+-sd':>18s}{'pinn3 mean+-sd':>18s}{'diff (pinn-base)':>18s}{'seeds PINN better':>19s}{'paired t p':>10s}")
    for k, name, hb in M:
        b = np.array([get(r, k) for r in R["base3"]]); p = np.array([get(r, k) for r in R["pinn3"]])
        dlt = p - b; better = int(np.sum(dlt > 0) if hb else np.sum(dlt < 0))
        pv = float(stats.ttest_rel(p, b).pvalue) if np.all(np.isfinite(dlt)) and np.std(dlt) > 0 else float("nan")
        print(f"{name:26s}{f'{b.mean():.4f}+-{b.std(ddof=1):.4f}':>18s}{f'{p.mean():.4f}+-{p.std(ddof=1):.4f}':>18s}"
              f"{dlt.mean():>+18.4f}{f'{better}/{len(SEEDS)}':>19s}{pv:>10.3f}")
        out["metrics"][k] = {"base3": b.tolist(), "pinn3": p.tolist(), "higher_is_better": hb, "paired_t_p": pv}
    print(f"{'physics residual real test':26s}{np.mean([r['physics_residual_real_test'] for r in R['base3']]):>18.4f}")
    cb = np.concatenate([np.load(f"{D}/samples_base3_s{s}_ep{EPOCHS:03d}.npz")["correct"] for s in SEEDS]).astype(bool)
    cp = np.concatenate([np.load(f"{D}/samples_pinn3_s{s}_ep{EPOCHS:03d}.npz")["correct"] for s in SEEDS]).astype(bool)
    b_, c_ = int((cb & ~cp).sum()), int((~cb & cp).sum()); n_ = b_ + c_
    p_mc = min(1.0, 2 * sum(math.comb(n_, k) for k in range(min(b_, c_) + 1)) / 2 ** n_) if n_ else 1.0
    print(f"\nPooled paired McNemar (oracle correctness, {len(cb)} pairs): base-only={b_}, pinn-only={c_}, p={p_mc:.4f}")
    out["mcnemar_pooled"] = {"b": b_, "c": c_, "p": p_mc}
    # downstream
    def auc_of(prefix):
        fs = sorted(glob.glob(f"{DS}/clf_{prefix}*_c*.json"))
        return np.array([json.load(open(f))["auc"] for f in fs]), np.array([json.load(open(f))["balanced_acc"] for f in fs])
    ar, br = auc_of("real_only"); ab, bb = auc_of("base3_s"); ap_, bp = auc_of("pinn3_s")
    print("\n" + "=" * 72 + "\nDOWNSTREAM (ResNet-18 trained on real 25% [+ 400 synthetic], tested on REAL test set)")
    for name, A_, B_ in (("real only", ar, br), ("real + base3 synthetic", ab, bb), ("real + pinn3 synthetic", ap_, bp)):
        print(f"{name:26s} AUC {A_.mean():.4f} +- {A_.std(ddof=1):.4f} | balanced acc {B_.mean():.4f} +- {B_.std(ddof=1):.4f} | n={len(A_)}")
    t_pb = stats.ttest_ind(ap_, ab, equal_var=False); t_br = stats.ttest_ind(ab, ar, equal_var=False)
    t_pr = stats.ttest_ind(ap_, ar, equal_var=False)
    print(f"Welch t-test AUC: pinn vs base p={t_pb.pvalue:.4f} | base vs real-only p={t_br.pvalue:.4f} | pinn vs real-only p={t_pr.pvalue:.4f}")
    out["downstream"] = {"real_only": ar.tolist(), "base3": ab.tolist(), "pinn3": ap_.tolist(),
                         "p_pinn_vs_base": float(t_pb.pvalue), "p_base_vs_real": float(t_br.pvalue), "p_pinn_vs_real": float(t_pr.pvalue)}
    # pre-registered verdict
    primary = bool(ap_.mean() > ab.mean() and t_pb.pvalue < 0.05)
    kid_all = all(np.array(out["metrics"]["KID_mean"]["pinn3"]) < np.array(out["metrics"]["KID_mean"]["base3"]))
    phys_better = np.mean(out["metrics"]["physics_abs_gap_to_real"]["pinn3"]) <= np.mean(out["metrics"]["physics_abs_gap_to_real"]["base3"])
    print("\n" + "=" * 72 + "\nPRE-REGISTERED VERDICT")
    print(f"PRIMARY  (downstream AUC: pinn > base, Welch p<0.05): {'MET' if primary else 'NOT MET'}")
    print(f"SECONDARY (Inception KID lower in all {len(SEEDS)} seeds):   {'MET' if kid_all else 'NOT MET'}")
    print(f"SECONDARY (physics at least as close to real):        {'MET' if phys_better else 'NOT MET'}")
    out["verdict"] = {"primary_met": primary, "kid_all_seeds": bool(kid_all), "physics_closer": bool(phys_better)}
    json.dump(out, open(f"{D}/final_summary.json", "w"), indent=2)
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    keys = [("KID_mean", "KID Inception (lower better)"), ("KID_mammo_mean", "KID mammography (lower better)"),
            ("physics_abs_gap_to_real", "|physics - real| (lower better)")]
    fig, ax = plt.subplots(1, 4, figsize=(18, 4))
    for i, (k, t) in enumerate(keys):
        for j, p in enumerate(("base3", "pinn3")):
            v = out["metrics"][k][p]; ax[i].scatter([j] * len(v), v, s=40); ax[i].bar(j, np.mean(v), alpha=0.3)
        ax[i].set_xticks([0, 1]); ax[i].set_xticklabels(["baseline", "PINN"]); ax[i].set_title(t)
    for j, (n, v) in enumerate((("real only", ar), ("+ baseline synth", ab), ("+ PINN synth", ap_))):
        ax[3].scatter([j] * len(v), v, s=40); ax[3].bar(j, v.mean(), alpha=0.3)
    ax[3].set_xticks([0, 1, 2]); ax[3].set_xticklabels(["real", "+base", "+PINN"]); ax[3].set_title("downstream AUC (higher better)")
    plt.tight_layout(); plt.savefig(f"{D}/final_figure.png", dpi=130); plt.show()
    print(f"\nSaved: {D}/final_summary.json and {D}/final_figure.png  (+ results_*.json, grid_*.png, {DS}/clf_*.json)")
